In [1]:
# ============================================
# Instalar PySpark (necessário no Colab)
# ============================================
!pip -q install pyspark

# ============================================
# Importar bibliotecas
# ============================================
import requests
from pyspark.sql import SparkSession

# ============================================
# Inicializar Spark
# ============================================
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Case Nexa Credito - SQL com Spark")
    .getOrCreate()
)

# ============================================
# URL base do bucket público
# ============================================
BASE_URL = "https://public-data-for-technical-interview.s3.amazonaws.com/dinamica_estag_analytics"

files = [
    "clientes.csv",
    "propostas_credito.csv",
    "contratos.csv",
    "parcelas.csv",
    "pagamentos.csv"
]

# ============================================
# Download dos arquivos
# ============================================
for file_name in files:
    url = f"{BASE_URL}/{file_name}"
    response = requests.get(url)
    response.raise_for_status()

    with open(file_name, "wb") as f:
        f.write(response.content)

    print(f"{file_name} baixado com sucesso.")

# ============================================
# Leitura dos CSVs com Spark
# ============================================
clientes = spark.read.csv(
    "clientes.csv",
    header=True,
    inferSchema=True
)

propostas = spark.read.csv(
    "propostas_credito.csv",
    header=True,
    inferSchema=True
)

contratos = spark.read.csv(
    "contratos.csv",
    header=True,
    inferSchema=True
)

parcelas = spark.read.csv(
    "parcelas.csv",
    header=True,
    inferSchema=True
)

pagamentos = spark.read.csv(
    "pagamentos.csv",
    header=True,
    inferSchema=True
)

# ============================================
# Criar views temporárias para SQL
# ============================================
clientes.createOrReplaceTempView("clientes")
propostas.createOrReplaceTempView("propostas_credito")
contratos.createOrReplaceTempView("contratos")
parcelas.createOrReplaceTempView("parcelas")
pagamentos.createOrReplaceTempView("pagamentos")

# ============================================
# Validação
# ============================================
print("Views criadas com sucesso:")
print("- clientes")
print("- propostas_credito")
print("- contratos")
print("- parcelas")
print("- pagamentos")

print("\nContagem de registros:")

spark.sql("SELECT COUNT(*) AS total FROM clientes").show()
spark.sql("SELECT COUNT(*) AS total FROM propostas_credito").show()
spark.sql("SELECT COUNT(*) AS total FROM contratos").show()
spark.sql("SELECT COUNT(*) AS total FROM parcelas").show()
spark.sql("SELECT COUNT(*) AS total FROM pagamentos").show()

clientes.csv baixado com sucesso.
propostas_credito.csv baixado com sucesso.
contratos.csv baixado com sucesso.
parcelas.csv baixado com sucesso.
pagamentos.csv baixado com sucesso.
Views criadas com sucesso:
- clientes
- propostas_credito
- contratos
- parcelas
- pagamentos

Contagem de registros:
+-----+
|total|
+-----+
|  502|
+-----+

+-----+
|total|
+-----+
|  903|
+-----+

+-----+
|total|
+-----+
|  486|
+-----+

+-----+
|total|
+-----+
| 4389|
+-----+

+-----+
|total|
+-----+
| 3326|
+-----+



In [2]:
clientes = spark.sql("""
SELECT
    *
FROM clientes
""")

clientes.show()

+----------+------------+------------------+-------------+--------------+---+-----------+-------------+------------+
|id_cliente|tipo_cliente|         documento|data_cadastro|        cidade| uf|   segmento|score_credito|renda_mensal|
+----------+------------+------------------+-------------+--------------+---+-----------+-------------+------------+
|     10000|          PF|   1.0433218196E10|   2024-05-26|  Porto Alegre| RS|        PME|          563|      5389.0|
|     10001|          PJ|  1.33890838637E11|   2025-05-23|     São Paulo| SP|    Premium|          410|     13186.9|
|     10002|          PF|   9.4026542351E10|   2024-07-17|     Fortaleza| CE|    Premium|          365|     4090.51|
|     10003|          PF|   1.6155940781E10|   2024-12-05|     São Paulo| SP|Mass Market|          675|     5534.87|
|     10004|          PF|   6.1849593103E10|   2025-11-20| Florianópolis| SC|        MEI|          575|     8898.96|
|     10005|          PF|   4.1316475255E10|   2024-02-18|      

In [3]:
propostas = spark.sql("""
SELECT
    *
FROM propostas_credito
""")

propostas.show()

+-----------+-------------+----------+-------------+---------------+----------------+-----------+----------------------+---------------+------------------+
|id_proposta|data_proposta|id_cliente| canal_origem|produto_credito|valor_solicitado|prazo_meses|taxa_mensal_solicitada|status_proposta|     motivo_recusa|
+-----------+-------------+----------+-------------+---------------+----------------+-----------+----------------------+---------------+------------------+
|     200000|   2025-07-31|     10101|          App|    Antecipação|         5089.36|         12|                0.0228|       APROVADA|              NULL|
|     200001|   2025-10-02|     10424|     WhatsApp|Capital de Giro|         6614.55|          6|                0.0271|       APROVADA|              NULL|
|     200002|   2025-10-22|     10391|   Televendas|      Parcelado|        19102.14|          9|                0.1063|       APROVADA|              NULL|
|     200003|   2025-10-03|     10266|     WhatsApp|      Parcel

In [4]:
contratos = spark.sql("""
SELECT
    *
FROM contratos
""")

contratos.show()

+-----------+-----------+----------------+--------------+-----------------+-----------------+---------------+----------------+---------------------+
|id_contrato|id_proposta|data_contratacao|valor_liberado|taxa_mensal_final|prazo_final_meses|status_contrato|canal_originacao|produto_credito_final|
+-----------+-----------+----------------+--------------+-----------------+-----------------+---------------+----------------+---------------------+
|   300000.0|     200300|      2025-07-04|      10543.01|           0.0145|               12|      LIQUIDADO|             App|            Parcelado|
|   300001.0|     200841|      2025-03-22|      29734.09|           0.0182|                9|      LIQUIDADO|             App|            Parcelado|
|   300002.0|     200183|      2025-04-20|      15944.48|           0.0337|               12|          ATIVO|             App|           Consignado|
|   300003.0|     200765|      2025-09-28|       6539.06|           0.0622|                6|          ATI

In [5]:
parcelas = spark.sql("""
SELECT
    *
FROM parcelas
""")

parcelas.show()

+----------+-----------+-----------+---------------+-------------+---------------+-----------+--------------+
|id_parcela|id_contrato|num_parcela|data_vencimento|valor_parcela|valor_principal|valor_juros|status_parcela|
+----------+-----------+-----------+---------------+-------------+---------------+-----------+--------------+
|  400000.0|     300000|          1|     2025-08-03|       910.32|          746.8|     131.79|        ABERTA|
|  400001.0|     300000|          2|     2025-09-02|       908.22|          746.8|     131.79|        ABERTA|
|  400002.0|     300000|          3|     2025-10-02|       891.41|          746.8|     131.79|        ABERTA|
|  400003.0|     300000|          4|     2025-11-01|       852.29|          746.8|     131.79|          PAGA|
|  400004.0|     300000|          5|     2025-12-01|       873.32|          746.8|     131.79|        ABERTA|
|  400005.0|     300000|          6|     2025-12-31|       877.39|          746.8|     131.79|        ABERTA|
|  400006.

In [6]:
pagamentos = spark.sql("""
SELECT
    *
FROM pagamentos
""")

pagamentos.show()

+------------+----------+--------------+----------+-----------+----------------+--------------+
|id_pagamento|id_parcela|data_pagamento|valor_pago|valor_multa|valor_juros_mora|valor_desconto|
+------------+----------+--------------+----------+-----------+----------------+--------------+
|      500000|    400003|    2025-11-01|    428.48|        0.0|             0.0|           0.0|
|      500001|    400003|    2025-11-06|    412.62|        0.0|             0.0|           0.0|
|      500002|    400006|    2026-02-04|    890.22|        0.0|             0.0|           0.0|
|      500003|    400007|    2026-03-03|    851.02|        0.0|             0.0|           0.0|
|      500004|    400009|    2026-05-05|    913.37|      20.07|            0.36|           0.0|
|      500005|    400010|    2026-06-01|    829.71|        0.0|             0.0|          9.45|
|      500006|    400011|    2026-07-19|    908.34|        0.0|             0.0|           0.0|
|      500007|    400012|    2025-04-22|

In [11]:
dim_cliente = spark.sql("""
WITH source_clientes AS (
    SELECT
        CAST(id_cliente AS BIGINT)                        AS id_cliente,
        UPPER(TRIM(tipo_cliente))                         AS tipo_cliente,
        CAST(CAST(documento AS BIGINT) AS STRING)         AS nr_documento,
        TO_DATE(data_cadastro)                            AS data_cadastro,
        TRIM(cidade)                                      AS nm_cidade,
        UPPER(TRIM(uf))                                   AS cd_uf,
        TRIM(segmento)                                    AS ds_segmento,
        CAST(score_credito AS INT)                        AS score_credito,
        CAST(renda_mensal AS DECIMAL(15, 2))              AS vl_renda_mensal
    FROM clientes
),

transform_dim AS (
    SELECT
        id_cliente,
        tipo_cliente,
        nr_documento,
        data_cadastro,
        nm_cidade,
        cd_uf,
        ds_segmento,
        score_credito,
        vl_renda_mensal,
        CASE
            WHEN score_credito < 500 THEN 'Alto Risco'
            WHEN score_credito BETWEEN 500 AND 699 THEN 'Médio Risco'
            ELSE 'Baixo Risco'
        END AS faixa_score,
        CASE
            WHEN vl_renda_mensal <= 3000 THEN 'Até 3k'
            WHEN vl_renda_mensal <= 7000 THEN '3k a 7k'
            WHEN vl_renda_mensal <= 15000 THEN '7k a 15k'
            ELSE 'Acima de 15k'
        END AS faixa_renda
    FROM source_clientes
)

SELECT * FROM transform_dim;
""")

dim_cliente.createOrReplaceTempView("dim_cliente")

print("Dimensão Clientes:\n")

spark.sql("SELECT COUNT(*) AS total FROM dim_cliente").show()

Dimensão Clientes:/n
+-----+
|total|
+-----+
|  502|
+-----+



In [12]:
dim_calendario = spark.sql("""
WITH date_series AS (
    SELECT
        EXPLODE(SEQUENCE(DATE('2020-01-01'), DATE('2030-12-31'), INTERVAL 1 DAY)) AS data_referencia
),

transform_calendario AS (
    SELECT
        CAST(DATE_FORMAT(data_referencia, 'yyyyMMdd') AS INT) AS id_data,
        data_referencia,
        YEAR(data_referencia)                                 AS nr_ano,
        MONTH(data_referencia)                                AS nr_mes,
        DAY(data_referencia)                                  AS nr_dia,
        DATE_FORMAT(data_referencia, 'yyyy-MM')               AS anomes,
        QUARTER(data_referencia)                              AS nr_trimestre,
        DAYOFWEEK(data_referencia)                            AS nr_dia_semana,
        CASE WHEN DAYOFWEEK(data_referencia) IN (1, 7) THEN TRUE ELSE FALSE END AS is_fim_de_semana
    FROM date_series
)

SELECT * FROM transform_calendario;
""")

dim_calendario.createOrReplaceTempView("dim_calendario")

print("Dimensão Calendário:\n")

spark.sql("SELECT COUNT(*) AS total FROM dim_calendario").show()

Dimensão Calendário:/n
+-----+
|total|
+-----+
| 4018|
+-----+



In [14]:
fato_propostas = spark.sql("""
WITH source_propostas AS (
    SELECT
        CAST(id_proposta AS BIGINT)                               AS id_proposta,
        TO_DATE(data_proposta)                                    AS data_proposta,
        CAST(id_cliente AS BIGINT)                                AS id_cliente,
        TRIM(canal_origem)                                        AS canal_origem,
        TRIM(produto_credito)                                     AS produto_credito,
        -- Limpeza de caracteres monetários e conversão para decimal
        CAST(
            REGEXP_REPLACE(
                REGEXP_REPLACE(valor_solicitado, '[R$\\s]', ''),
                ',', '.'
            ) AS DECIMAL(15, 2)
        )                                                         AS vl_solicitado,
        CAST(prazo_meses AS INT)                                  AS prazo_meses_solicitado,
        CAST(taxa_mensal_solicitada AS DECIMAL(8, 4))             AS tx_mensal_solicitada,
        UPPER(TRIM(status_proposta))                              AS status_proposta,
        NULLIF(TRIM(motivo_recusa), '')                           AS motivo_recusa
    FROM propostas_credito
),

transform_fato AS (
    SELECT
        p.id_proposta,
        CAST(DATE_FORMAT(p.data_proposta, 'yyyyMMdd') AS INT)     AS sk_data_proposta,
        p.data_proposta,
        p.id_cliente,
        p.canal_origem,
        p.produto_credito,
        p.vl_solicitado,
        p.prazo_meses_solicitado,
        p.tx_mensal_solicitada,
        p.status_proposta,
        p.motivo_recusa,
        CASE WHEN p.status_proposta = 'APROVADA' THEN 1 ELSE 0 END   AS is_aprovada,
        CASE WHEN p.status_proposta = 'RECUSADA' THEN 1 ELSE 0 END   AS is_recusada,
        CASE WHEN p.status_proposta = 'EM_ANALISE' THEN 1 ELSE 0 END AS is_em_analise
    FROM source_propostas p
)

SELECT * FROM transform_fato;
""")

fato_propostas.createOrReplaceTempView("fato_propostas")

print("Fato Propostas:\n")

spark.sql("SELECT COUNT(*) AS total FROM fato_propostas").show()

Fato Propostas:/n
+-----+
|total|
+-----+
|  903|
+-----+



In [15]:
fato_contratos = spark.sql("""
WITH source_contratos AS (
    SELECT
        CAST(id_contrato AS BIGINT)                               AS id_contrato,
        CAST(id_proposta AS BIGINT)                               AS id_proposta,
        TO_DATE(data_contratacao)                                 AS data_contratacao,
        CAST(valor_liberado AS DECIMAL(15, 2))                    AS vl_liberado,
        CAST(taxa_mensal_final AS DECIMAL(8, 4))                  AS tx_mensal_final,
        CAST(prazo_final_meses AS INT)                            AS prazo_final_meses,
        UPPER(TRIM(status_contrato))                              AS status_contrato,
        TRIM(canal_originacao)                                    AS canal_originacao,
        TRIM(produto_credito_final)                               AS produto_credito_final
    FROM contratos
),

transform_fato AS (
    SELECT
        c.id_contrato,
        c.id_proposta,
        CAST(DATE_FORMAT(c.data_contratacao, 'yyyyMMdd') AS INT)  AS sk_data_contratacao,
        c.data_contratacao,
        c.vl_liberado,
        c.tx_mensal_final,
        c.prazo_final_meses,
        c.status_contrato,
        c.canal_originacao,
        c.produto_credito_final,
        CAST(
            (c.vl_liberado * (1 + (c.tx_mensal_final * c.prazo_final_meses)))
            AS DECIMAL(15, 2)
        )                                                         AS vl_estimado_total
    FROM source_contratos c
)

SELECT * FROM transform_fato;
""")

fato_contratos.createOrReplaceTempView("fato_contratos")

print("Fato Contratos:\n")

spark.sql("SELECT COUNT(*) AS total FROM fato_contratos").show()

Fato Contratos:/n
+-----+
|total|
+-----+
|  486|
+-----+



In [16]:
fato_parcelas = spark.sql("""
WITH source_parcelas AS (
    SELECT
        CAST(id_parcela AS BIGINT)                               AS id_parcela,
        CAST(id_contrato AS BIGINT)                              AS id_contrato,
        CAST(num_parcela AS INT)                                 AS num_parcela,
        TO_DATE(data_vencimento)                                 AS data_vencimento,
        CAST(valor_parcela AS DECIMAL(15, 2))                    AS vl_parcela,
        CAST(valor_principal AS DECIMAL(15, 2))                  AS vl_principal,
        CAST(valor_juros AS DECIMAL(15, 2))                      AS vl_juros,
        UPPER(TRIM(status_parcela))                              AS status_parcela
    FROM parcelas
),

transform_fato AS (
    SELECT
        p.id_parcela,
        p.id_contrato,
        p.num_parcela,
        CAST(DATE_FORMAT(p.data_vencimento, 'yyyyMMdd') AS INT)  AS sk_data_vencimento,
        p.data_vencimento,
        p.vl_parcela,
        p.vl_principal,
        p.vl_juros,
        p.status_parcela,
        CASE WHEN p.status_parcela = 'PAGA' THEN 1 ELSE 0 END      AS is_paga,
        CASE WHEN p.status_parcela = 'EM_ATRASO' THEN 1 ELSE 0 END AS is_em_atraso,
        CASE WHEN p.status_parcela = 'ABERTA' THEN 1 ELSE 0 END    AS is_aberta
    FROM source_parcelas p
)

SELECT * FROM transform_fato;
""")

fato_parcelas.createOrReplaceTempView("fato_parcelas")

print("Fato Parcelas:\n")

spark.sql("SELECT COUNT(*) AS total FROM fato_parcelas").show()

Fato Parcelas:/n
+-----+
|total|
+-----+
| 4389|
+-----+



In [17]:
fato_pagamentos = spark.sql("""
WITH source_pagamentos AS (
    SELECT
        CAST(id_pagamento AS BIGINT)                             AS id_pagamento,
        CAST(id_parcela AS BIGINT)                               AS id_parcela,
        TO_DATE(data_pagamento)                                  AS data_pagamento,
        -- Conversão da string monetária para decimal
        CAST(
            REGEXP_REPLACE(
                REGEXP_REPLACE(valor_pago, '[R$\\s]', ''),
                ',', '.'
            ) AS DECIMAL(15, 2)
        )                                                        AS vl_pago,
        COALESCE(CAST(valor_multa AS DECIMAL(15, 2)), 0.00)      AS vl_multa,
        COALESCE(CAST(valor_juros_mora AS DECIMAL(15, 2)), 0.00) AS vl_juros_mora,
        COALESCE(CAST(valor_desconto AS DECIMAL(15, 2)), 0.00)   AS vl_desconto
    FROM pagamentos
),

transform_fato AS (
    SELECT
        pg.id_pagamento,
        pg.id_parcela,
        CAST(DATE_FORMAT(pg.data_pagamento, 'yyyyMMdd') AS INT)  AS sk_data_pagamento,
        pg.data_pagamento,
        pg.vl_pago,
        pg.vl_multa,
        pg.vl_juros_mora,
        pg.vl_desconto,
        (pg.vl_multa + pg.vl_juros_mora - pg.vl_desconto)        AS vl_resultado_encargos
    FROM source_pagamentos pg
)

SELECT * FROM transform_fato;
""")

fato_pagamentos.createOrReplaceTempView("fato_pagamentos")

print("Fato Pagamentos:\n")

spark.sql("SELECT COUNT(*) AS total FROM fato_pagamentos").show()

Fato Pagamentos:/n
+-----+
|total|
+-----+
| 3326|
+-----+



### 1. Quantas propostas foram recebidas?
* **Objetivo:** Medir a demanda bruta do topo do funil e a capacidade de atração dos canais de aquisição.
* **Explicação:** Conta o total de propostas cadastradas na esteira, servindo como denominador base para qualquer cálculo de conversão.
* **Como calcular:** Contagem simples de `id_proposta` na tabela de fatos de propostas.

In [18]:
spark.sql("""
SELECT
    COUNT(id_proposta) AS total_propostas_recebidas
FROM fato_propostas
""").show()

+-------------------------+
|total_propostas_recebidas|
+-------------------------+
|                      903|
+-------------------------+



### 2. Quanto crédito foi concedido em contratos?
* **Objetivo:** Avaliar o volume financeiro real originado e injetado no mercado.
* **Explicação:** Soma o `vl_liberado` dos contratos efetivados, desconsiderando contratos cancelados para não inflar a carteira ativa.
* **Como calcular:** Soma de `vl_liberado` com filtro de status diferente de `'CANCELADO'`.

In [19]:
spark.sql("""
SELECT
    SUM(vl_liberado) AS total_credito_concedido,
    COUNT(id_contrato) AS total_contratos_gerados
FROM fato_contratos
WHERE status_contrato <> 'CANCELADO'
""").show()

+-----------------------+-----------------------+
|total_credito_concedido|total_contratos_gerados|
+-----------------------+-----------------------+
|             7071355.53|                    455|
+-----------------------+-----------------------+



### 3. Quanto já foi pago pelos clientes?
* **Objetivo:** Acompanhar a liquidez e a recuperação de caixa realizada.
* **Explicação:** Mede o montante total liquidado em dinheiro pelos tomadores de crédito em eventos de pagamento.
* **Como calcular:** Soma do campo `vl_pago` da `fato_pagamentos`.

In [20]:
spark.sql("""
SELECT
    SUM(vl_pago) AS total_valor_pago,
    COUNT(id_pagamento) AS total_eventos_pagamento
FROM fato_pagamentos
""").show()

+----------------+-----------------------+
|total_valor_pago|total_eventos_pagamento|
+----------------+-----------------------+
|      4968562.55|                   3326|
+----------------+-----------------------+



### 4. Qual é a taxa de conversão global do funil (Proposta → Contrato)?
* **Objetivo:** Medir a eficiência de ponta a ponta da esteira de crédito.
* **Explicação:** Mostra qual fatia percentual da demanda inicial realmente virou receita e contrato ativo para a fintech.
* **Como calcular:** Razão entre a contagem de contratos válidos e a contagem de propostas totais.

In [21]:
spark.sql("""
SELECT
    COUNT(DISTINCT p.id_proposta) AS total_propostas,
    COUNT(DISTINCT c.id_contrato) AS total_contratos,
    ROUND(COUNT(DISTINCT c.id_contrato) * 100.0 / COUNT(DISTINCT p.id_proposta), 2) AS taxa_conversao_pct
FROM fato_propostas p
LEFT JOIN fato_contratos c
    ON p.id_proposta = c.id_proposta
   AND c.status_contrato <> 'CANCELADO'
""").show()

+---------------+---------------+------------------+
|total_propostas|total_contratos|taxa_conversao_pct|
+---------------+---------------+------------------+
|            901|            452|             50.17|
+---------------+---------------+------------------+



### 5. Qual é o ticket médio e o prazo médio dos contratos concedidos?
* **Objetivo:** Entender o porte médio das operações de crédito e a duração média da exposição ao risco.
* **Explicação:** Apoia a tesouraria no planejamento de liquidez e precificação de linhas de crédito.
* **Como calcular:** Média aritmética de `vl_liberado`, `prazo_final_meses` e `tx_mensal_final`.

In [22]:
spark.sql("""
SELECT
    ROUND(AVG(vl_liberado), 2) AS ticket_medio_concedido,
    ROUND(AVG(prazo_final_meses), 1) AS prazo_medio_meses,
    ROUND(AVG(tx_mensal_final) * 100, 2) AS taxa_media_mensal_pct
FROM fato_contratos
WHERE status_contrato <> 'CANCELADO'
""").show()

+----------------------+-----------------+---------------------+
|ticket_medio_concedido|prazo_medio_meses|taxa_media_mensal_pct|
+----------------------+-----------------+---------------------+
|              15541.44|             12.0|                 6.03|
+----------------------+-----------------+---------------------+



### 6. Quais são os principais motivos de recusa de propostas?
* **Objetivo:** Identificar as principais fricções e barreiras de entrada na política de crédito.
* **Explicação:** Quantifica os motivos de reprovação para orientar eventuais flexibilizações ou ajustes de segmentação de marketing.
* **Como calcular:** Agrupamento por `motivo_recusa` filtrando apenas propostas reprovadas.

In [23]:
spark.sql("""
SELECT
    COALESCE(motivo_recusa, 'Não Informado') AS motivo_recusa,
    COUNT(id_proposta) AS qtd_propostas_recusadas,
    ROUND(COUNT(id_proposta) * 100.0 / SUM(COUNT(id_proposta)) OVER(), 2) AS share_recusa_pct
FROM fato_propostas
WHERE is_recusada = 1
GROUP BY motivo_recusa
ORDER BY qtd_propostas_recusadas DESC
""").show(truncate=False)

+------------------+-----------------------+----------------+
|motivo_recusa     |qtd_propostas_recusadas|share_recusa_pct|
+------------------+-----------------------+----------------+
|Renda insuficiente|81                     |24.25           |
|Política          |73                     |21.86           |
|Score baixo       |67                     |20.06           |
|Fraude            |56                     |16.77           |
|Documentação      |55                     |16.47           |
|Não Informado     |2                      |0.60            |
+------------------+-----------------------+----------------+



### 7. Qual canal de origem apresenta a melhor eficiência de conversão e menor atrito?
* **Objetivo:** Direcionar investimentos de aquisição para os canais mais saudáveis operacionalmente.
* **Explicação:** Cruza canais de entrada com taxas de aprovação e cancelamento, identificando canais que trazem muito volume, mas baixo aproveitamento.
* **Como calcular:** Agrupamento por `canal_origem` com contagens condicionais de propostas e contratos.

In [24]:
spark.sql("""
SELECT
    p.canal_origem,
    COUNT(p.id_proposta) AS volume_propostas,
    SUM(p.is_aprovada) AS propostas_aprovadas,
    ROUND(SUM(p.is_aprovada) * 100.0 / COUNT(p.id_proposta), 2) AS taxa_aprovacao_pct,
    COUNT(c.id_contrato) AS contratos_efetivados,
    ROUND(COUNT(c.id_contrato) * 100.0 / COUNT(p.id_proposta), 2) AS taxa_conversao_final_pct
FROM fato_propostas p
LEFT JOIN fato_contratos c
    ON p.id_proposta = c.id_proposta
   AND c.status_contrato <> 'CANCELADO'
GROUP BY p.canal_origem
ORDER BY taxa_conversao_final_pct DESC
""").show()

+-------------+----------------+-------------------+------------------+--------------------+------------------------+
| canal_origem|volume_propostas|propostas_aprovadas|taxa_aprovacao_pct|contratos_efetivados|taxa_conversao_final_pct|
+-------------+----------------+-------------------+------------------+--------------------+------------------------+
|     WhatsApp|             138|                 76|             55.07|                  71|                   51.45|
|    Indicação|              82|                 45|             54.88|                  42|                   51.22|
|   Televendas|             102|                 58|             56.86|                  52|                   50.98|
|          App|             402|                217|             53.98|                 201|                   50.00|
|Loja Parceira|             179|                 96|             53.63|                  87|                   48.60|
+-------------+----------------+-------------------+----

### 8. Existe dispersão significativa entre o valor solicitado pelo cliente e o valor liberado?
* **Objetivo:** Medir se a política de corte de limite (*haircut*) desestimula a assinatura do contrato.
* **Explicação:** Se a fintech concede um valor muito inferior ao solicitado, o cliente pode desistir de assinar o contrato final.
* **Como calcular:** Comparação entre `vl_solicitado` e `vl_liberado` por produto.

In [25]:
spark.sql("""
SELECT
    p.produto_credito,
    COUNT(c.id_contrato) AS total_contratado,
    ROUND(AVG(p.vl_solicitado), 2) AS media_solicitada,
    ROUND(AVG(c.vl_liberado), 2) AS media_liberada,
    ROUND(AVG(c.vl_liberado - p.vl_solicitado), 2) AS diferenca_absoluta_media,
    ROUND(AVG((c.vl_liberado / p.vl_solicitado) - 1) * 100, 2) AS variacao_percentual_media
FROM fato_propostas p
INNER JOIN fato_contratos c ON p.id_proposta = c.id_proposta
WHERE c.status_contrato <> 'CANCELADO'
GROUP BY p.produto_credito
""").show()

+---------------+----------------+----------------+--------------+------------------------+-------------------------+
|produto_credito|total_contratado|media_solicitada|media_liberada|diferenca_absoluta_media|variacao_percentual_media|
+---------------+----------------+----------------+--------------+------------------------+-------------------------+
|    Antecipação|              77|        14253.72|      16107.29|                 1853.56|                   116.76|
|      Parcelado|             238|        14944.15|      15528.86|                  584.71|                   120.93|
|Capital de Giro|             101|        14215.50|      15736.35|                 1520.85|                   193.79|
|     Consignado|              37|        15018.39|      14186.63|                 -831.76|                   138.19|
+---------------+----------------+----------------+--------------+------------------------+-------------------------+



### 9. A taxa de juros praticada é condizente com o prazo e com o produto?
* **Objetivo:** Analisar a curva de precificação por produto e duração do empréstimo.
* **Explicação:** Permite identificar distorções onde produtos com prazos longos (maior risco de liquidez) estão sendo oferecidos com taxas iguais ou inferiores às de curto prazo.
* **Como calcular:** Média de taxa mensal agrupada por produto e faixa de prazo.

In [26]:
spark.sql("""
SELECT
    c.produto_credito_final,
    CASE
        WHEN c.prazo_final_meses <= 12 THEN 'Curto Prazo (até 12m)'
        WHEN c.prazo_final_meses <= 36 THEN 'Médio Prazo (13m a 36m)'
        ELSE 'Longo Prazo (> 36m)'
    END AS faixa_prazo,
    COUNT(c.id_contrato) AS qtd_contratos,
    ROUND(SUM(c.vl_liberado), 2) AS volume_liberado,
    ROUND(AVG(c.tx_mensal_final) * 100, 3) AS taxa_media_mensal_pct
FROM fato_contratos c
WHERE c.status_contrato <> 'CANCELADO'
GROUP BY 1, 2
ORDER BY c.produto_credito_final, faixa_prazo
""").show()

+---------------------+--------------------+-------------+---------------+---------------------+
|produto_credito_final|         faixa_prazo|qtd_contratos|volume_liberado|taxa_media_mensal_pct|
+---------------------+--------------------+-------------+---------------+---------------------+
|          Antecipação|Curto Prazo (até ...|           52|      835100.98|                6.067|
|          Antecipação|Médio Prazo (13m ...|           26|      419971.40|                6.540|
|      Capital de Giro|Curto Prazo (até ...|           67|     1062580.30|                5.824|
|      Capital de Giro|Médio Prazo (13m ...|           34|      522386.45|                5.829|
|           Consignado|Curto Prazo (até ...|           24|      313911.18|                5.774|
|           Consignado|Médio Prazo (13m ...|           13|      210994.19|                6.282|
|            Parcelado|Curto Prazo (até ...|          180|     2890648.24|                5.978|
|            Parcelado|Médio P

### 10. A pontuação de score de crédito reflete a inadimplência observada na prática?
* **Objetivo:** Avaliar a calibração do modelo de score da esteira de risco.
* **Explicação:** Calcula o percentual de parcelas atrasadas por faixa de score. Faixas de maior risco deveriam concentrar a maior parte do atraso; caso contrário, as regras de corte precisam ser revistas.
* **Como calcular:** Cruzar `fato_parcelas` com `dim_cliente` (através dos contratos e propostas) agrupando por `faixa_score`.

In [27]:
spark.sql("""
SELECT
    cli.faixa_score,
    COUNT(par.id_parcela) AS total_parcelas_vencidas_ou_pagas,
    SUM(par.is_em_atraso) AS parcelas_em_atraso,
    ROUND(SUM(par.is_em_atraso) * 100.0 / COUNT(par.id_parcela), 2) AS taxa_inadimplencia_parcelas_pct,
    ROUND(SUM(CASE WHEN par.is_em_atraso = 1 THEN par.vl_parcela ELSE 0 END), 2) AS saldo_financeiro_em_atraso
FROM fato_parcelas par
INNER JOIN fato_contratos c ON par.id_contrato = c.id_contrato
INNER JOIN fato_propostas p ON c.id_proposta = p.id_proposta
INNER JOIN dim_cliente cli ON p.id_cliente = cli.id_cliente
WHERE par.status_parcela IN ('EM_ATRASO', 'PAGA')
GROUP BY cli.faixa_score
ORDER BY taxa_inadimplencia_parcelas_pct DESC
""").show()

+-----------+--------------------------------+------------------+-------------------------------+--------------------------+
|faixa_score|total_parcelas_vencidas_ou_pagas|parcelas_em_atraso|taxa_inadimplencia_parcelas_pct|saldo_financeiro_em_atraso|
+-----------+--------------------------------+------------------+-------------------------------+--------------------------+
| Alto Risco|                            1397|                 0|                           0.00|                      0.00|
|Médio Risco|                             788|                 0|                           0.00|                      0.00|
|Baixo Risco|                             110|                 0|                           0.00|                      0.00|
+-----------+--------------------------------+------------------+-------------------------------+--------------------------+



### 11. Clientes com alto comprometimento de renda apresentam maior taxa de atraso?
* **Objetivo:** Validar a regra de corte de comprometimento de renda (parcela / renda).
* **Explicação:** Mostra a correlação entre o peso da parcela sobre os rendimentos do tomador e a sua pontualidade no pagamento.
* **Como calcular:** Calcular o percentual da renda comprometida pela parcela e tabular a taxa de atraso por faixas de comprometimento.

In [28]:
spark.sql("""
WITH parcelas_comprometimento AS (
    SELECT
        par.id_parcela,
        par.is_em_atraso,
        cli.vl_renda_mensal,
        par.vl_parcela,
        ROUND((par.vl_parcela / NULLIF(cli.vl_renda_mensal, 0)) * 100, 2) AS pct_comprometimento_renda
    FROM fato_parcelas par
    INNER JOIN fato_contratos c ON par.id_contrato = c.id_contrato
    INNER JOIN fato_propostas p ON c.id_proposta = p.id_proposta
    INNER JOIN dim_cliente cli ON p.id_cliente = cli.id_cliente
    WHERE par.status_parcela IN ('EM_ATRASO', 'PAGA')
      AND cli.vl_renda_mensal > 0
)
SELECT
    CASE
        WHEN pct_comprometimento_renda <= 15 THEN 'Até 15% da Renda'
        WHEN pct_comprometimento_renda <= 30 THEN '15% a 30% da Renda'
        WHEN pct_comprometimento_renda <= 50 THEN '30% a 50% da Renda'
        ELSE 'Acima de 50% da Renda'
    END AS faixa_comprometimento,
    COUNT(id_parcela) AS qtd_parcelas,
    SUM(is_em_atraso) AS qtd_em_atraso,
    ROUND(SUM(is_em_atraso) * 100.0 / COUNT(id_parcela), 2) AS taxa_inadimplencia_pct
FROM parcelas_comprometimento
GROUP BY 1
ORDER BY taxa_inadimplencia_pct DESC
""").show()

+---------------------+------------+-------------+----------------------+
|faixa_comprometimento|qtd_parcelas|qtd_em_atraso|taxa_inadimplencia_pct|
+---------------------+------------+-------------+----------------------+
|   15% a 30% da Renda|         460|            0|                  0.00|
|     Até 15% da Renda|         693|            0|                  0.00|
| Acima de 50% da R...|         747|            0|                  0.00|
|   30% a 50% da Renda|         395|            0|                  0.00|
+---------------------+------------+-------------+----------------------+



### 12. Qual segmento de clientes traz o melhor equilíbrio entre volume concedido e adimplência?
* **Objetivo:** Identificar o segmento ideal (ex.: PME, MEI, Mass Market, Premium) para focar expansão comercial.
* **Explicação:** Cruza a receita gerada em volume concedido com o percentual de adimplência de cada segmento.
* **Como calcular:** Agregar volume liberado e percentual de parcelas pagas por `ds_segmento`.

In [29]:
spark.sql("""
SELECT
    cli.ds_segmento,
    cli.tipo_cliente,
    COUNT(DISTINCT c.id_contrato) AS total_contratos,
    ROUND(SUM(c.vl_liberado), 2) AS volume_credito_concedido,
    ROUND(
        SUM(par.is_paga) * 100.0 / NULLIF(COUNT(par.id_parcela), 0), 2
    ) AS taxa_parcelas_pagas_pct
FROM dim_cliente cli
INNER JOIN fato_propostas p ON cli.id_cliente = p.id_cliente
INNER JOIN fato_contratos c ON p.id_proposta = c.id_proposta
LEFT JOIN fato_parcelas par ON c.id_contrato = par.id_contrato
WHERE c.status_contrato <> 'CANCELADO'
GROUP BY cli.ds_segmento, cli.tipo_cliente
ORDER BY volume_credito_concedido DESC
""").show()

+-----------+------------+---------------+------------------------+-----------------------+
|ds_segmento|tipo_cliente|total_contratos|volume_credito_concedido|taxa_parcelas_pagas_pct|
+-----------+------------+---------------+------------------------+-----------------------+
|        PME|          PF|             71|             10182422.10|                  54.43|
|Mass Market|          PF|             61|              9078058.55|                  55.81|
|        MEI|          PF|             63|              8818843.26|                  55.48|
|    Premium|          PF|             71|              8684724.41|                  53.98|
|  Emergente|          PF|             67|              8516999.17|                  56.77|
|    Premium|          PJ|             30|              5182002.33|                  57.54|
|Mass Market|          PJ|             30|              4051004.62|                  54.67|
|  Emergente|          PJ|             20|              3755092.95|             

### 13. Qual o impacto financeiro líquido da cobrança de encargos (multa/mora vs. descontos)?
* **Objetivo:** Medir a efetividade e o resultado financeiro da área de recuperação e cobrança.
* **Explicação:** Mostra se a empresa arrecada mais com juros e multas de atraso do que concede em abatimentos comerciais para quitação.
* **Como calcular:** Balanço entre `vl_multa`, `vl_juros_mora` e `vl_desconto` da `fato_pagamentos`.

In [30]:
spark.sql("""
SELECT
    COUNT(id_pagamento) AS total_pagamentos,
    ROUND(SUM(vl_pago), 2) AS total_recebido_principal,
    ROUND(SUM(vl_multa), 2) AS total_multa_arrecadada,
    ROUND(SUM(vl_juros_mora), 2) AS total_juros_mora_arrecadado,
    ROUND(SUM(vl_desconto), 2) AS total_desconto_concedido,
    ROUND(SUM(vl_resultado_encargos), 2) AS resultado_liquido_encargos
FROM fato_pagamentos
""").show()

+----------------+------------------------+----------------------+---------------------------+------------------------+--------------------------+
|total_pagamentos|total_recebido_principal|total_multa_arrecadada|total_juros_mora_arrecadado|total_desconto_concedido|resultado_liquido_encargos|
+----------------+------------------------+----------------------+---------------------------+------------------------+--------------------------+
|            3326|              4968562.55|              11977.54|                    7711.91|                 4583.47|                  15105.98|
+----------------+------------------------+----------------------+---------------------------+------------------------+--------------------------+



## Construção do OBT 1: Funil de Originação (`obt_funil_originacao`)
* **Grão analítico:** 1 linha por proposta de crédito recebida.
* **Objetivo:** Alimentar análises de conversão, taxa de aprovação, motivos de recusa, canais de aquisição e volume de crédito contratado.
* **Estratégia:** Une `fato_propostas` com `dim_cliente` e `fato_contratos` (via `LEFT JOIN`), garantindo que propostas reprovadas ou desistentes não sejam descartadas e mantendo a integridade métrica do funil.

In [35]:
obt_funil_originacao = spark.sql("""
WITH base_propostas AS (
    SELECT
        p.id_proposta,
        p.data_proposta,
        p.sk_data_proposta,
        p.id_cliente,
        p.canal_origem,
        p.produto_credito,
        p.vl_solicitado,
        p.prazo_meses_solicitado,
        p.tx_mensal_solicitada,
        p.status_proposta,
        p.motivo_recusa,
        p.is_aprovada,
        p.is_recusada,
        p.is_em_analise
    FROM fato_propostas p
),

enriquecimento_cliente AS (
    SELECT
        bp.*,
        c.tipo_cliente,
        c.nr_documento,
        c.nm_cidade,
        c.cd_uf,
        c.ds_segmento,
        c.score_credito,
        c.faixa_score,
        c.vl_renda_mensal,
        c.faixa_renda
    FROM base_propostas bp
    LEFT JOIN dim_cliente c ON bp.id_cliente = c.id_cliente
),

enriquecimento_contrato AS (
    SELECT
        ec.*,
        ct.id_contrato,
        ct.data_contratacao,
        ct.sk_data_contratacao,
        ct.vl_liberado,
        ct.tx_mensal_final,
        ct.prazo_final_meses,
        ct.status_contrato,
        ct.produto_credito_final,
        ct.vl_estimado_total,
        CASE WHEN ct.id_contrato IS NOT NULL THEN 1 ELSE 0 END AS is_contratado
    FROM enriquecimento_cliente ec
    LEFT JOIN fato_contratos ct ON ec.id_proposta = ct.id_proposta
)

SELECT * FROM enriquecimento_contrato;
""")

obt_funil_originacao.createOrReplaceTempView("obt_funil_originacao")

print("OBT Funil de Originação:\n")
spark.sql("SELECT * FROM obt_funil_originacao LIMIT 5").show(truncate=False)

OBT Funil de Originação:

+-----------+-------------+----------------+----------+------------+---------------+-------------+----------------------+--------------------+---------------+-------------+-----------+-----------+-------------+------------+------------+-------------+-----+-----------+-------------+-----------+---------------+-----------+-----------+----------------+-------------------+-----------+---------------+-----------------+---------------+---------------------+-----------------+-------------+
|id_proposta|data_proposta|sk_data_proposta|id_cliente|canal_origem|produto_credito|vl_solicitado|prazo_meses_solicitado|tx_mensal_solicitada|status_proposta|motivo_recusa|is_aprovada|is_recusada|is_em_analise|tipo_cliente|nr_documento|nm_cidade    |cd_uf|ds_segmento|score_credito|faixa_score|vl_renda_mensal|faixa_renda|id_contrato|data_contratacao|sk_data_contratacao|vl_liberado|tx_mensal_final|prazo_final_meses|status_contrato|produto_credito_final|vl_estimado_total|is_contratado

## Construção do OBT 2: Carteira & Cobrança (`obt_carteira_cobranca`)
* **Grão analítico:** 1 linha por parcela emitida.
* **Objetivo:** Alimentar dashboards e análises de cobrança, cronograma de vencimentos, inadimplência e recuperação de caixa.
* **Estratégia:** Consolida previamente os pagamentos por parcela em uma CTE para eliminar duplicidade de registros (*fan-out trap*), conectando em seguida os atributos de contrato, proposta e cliente.

In [36]:
obt_carteira_cobranca = spark.sql("""
WITH pagamentos_por_parcela AS (
    SELECT
        id_parcela,
        MAX(data_pagamento)           AS data_ultimo_pagamento,
        SUM(vl_pago)                  AS total_vl_pago,
        SUM(vl_multa)                 AS total_vl_multa,
        SUM(vl_juros_mora)            AS total_vl_mora,
        SUM(vl_desconto)              AS total_vl_desconto,
        SUM(vl_resultado_encargos)    AS total_resultado_encargos,
        COUNT(id_pagamento)           AS qtd_eventos_pagamento
    FROM fato_pagamentos
    GROUP BY id_parcela
),

parcelas_consolidadas AS (
    SELECT
        par.id_parcela,
        par.id_contrato,
        par.num_parcela,
        par.data_vencimento,
        par.sk_data_vencimento,
        par.vl_parcela,
        par.vl_principal,
        par.vl_juros,
        par.status_parcela,
        par.is_paga,
        par.is_em_atraso,
        par.is_aberta,
        COALESCE(pg.total_vl_pago, 0.00)              AS vl_pago_acumulado,
        COALESCE(pg.total_vl_multa, 0.00)             AS vl_multa_acumulada,
        COALESCE(pg.total_vl_mora, 0.00)              AS vl_mora_acumulada,
        COALESCE(pg.total_vl_desconto, 0.00)          AS vl_desconto_acumulado,
        COALESCE(pg.total_resultado_encargos, 0.00)   AS vl_resultado_encargos_acumulado,
        pg.data_ultimo_pagamento,
        COALESCE(pg.qtd_eventos_pagamento, 0)         AS qtd_pagamentos
    FROM fato_parcelas par
    LEFT JOIN pagamentos_por_parcela pg ON par.id_parcela = pg.id_parcela
),

enriquecimento_completo AS (
    SELECT
        pc.*,
        c.id_proposta,
        c.data_contratacao,
        c.sk_data_contratacao,
        c.vl_liberado                                  AS vl_liberado_contrato,
        c.tx_mensal_final,
        c.prazo_final_meses,
        c.status_contrato,
        c.canal_originacao,
        c.produto_credito_final,
        cli.id_cliente,
        cli.tipo_cliente,
        cli.nm_cidade,
        cli.cd_uf,
        cli.ds_segmento,
        cli.score_credito,
        cli.faixa_score,
        cli.vl_renda_mensal,
        cli.faixa_renda
    FROM parcelas_consolidadas pc
    INNER JOIN fato_contratos c ON pc.id_contrato = c.id_contrato
    INNER JOIN fato_propostas p ON c.id_proposta = p.id_proposta
    INNER JOIN dim_cliente cli ON p.id_cliente = cli.id_cliente
)

SELECT * FROM enriquecimento_completo;
""")

obt_carteira_cobranca.createOrReplaceTempView("obt_carteira_cobranca")

print("OBT Carteira de Cobrança:\n")
spark.sql("SELECT * FROM obt_carteira_cobranca LIMIT 5").show(truncate=False)

OBT Carteira de Cobrança:

+----------+-----------+-----------+---------------+------------------+----------+------------+--------+--------------+-------+------------+---------+-----------------+------------------+-----------------+---------------------+-------------------------------+---------------------+--------------+-----------+----------------+-------------------+--------------------+---------------+-----------------+---------------+----------------+---------------------+----------+------------+-------------+-----+-----------+-------------+-----------+---------------+-----------+
|id_parcela|id_contrato|num_parcela|data_vencimento|sk_data_vencimento|vl_parcela|vl_principal|vl_juros|status_parcela|is_paga|is_em_atraso|is_aberta|vl_pago_acumulado|vl_multa_acumulada|vl_mora_acumulada|vl_desconto_acumulado|vl_resultado_encargos_acumulado|data_ultimo_pagamento|qtd_pagamentos|id_proposta|data_contratacao|sk_data_contratacao|vl_liberado_contrato|tx_mensal_final|prazo_final_meses|status_

## Exportação das Tabelas Analíticas para o Power BI
* As tabelas são exportadas com `coalesce(1)` para gerar arquivos únicos consolidados.
* **Parquet:** Formato colunar recomendado para carregamento veloz no motor VertiPaq do Power BI, preservando metadados de tipagem (datas e decimais).
* **CSV:** Alternativa em formato delimitado por vírgula com cabeçalho ativado.
* Os arquivos ficam salvos no diretório local do Colab e podem ser baixados diretamente para sua máquina.

In [37]:
import os

output_dir_parquet = "/content/powerbi_export/parquet"
output_dir_csv = "/content/powerbi_export/csv"

os.makedirs(output_dir_parquet, exist_ok=True)
os.makedirs(output_dir_csv, exist_ok=True)

# 1. Exportação em Parquet (Recomendado)
spark.table("obt_funil_originacao").coalesce(1).write.mode("overwrite").parquet(
    f"{output_dir_parquet}/obt_funil_originacao"
)
spark.table("obt_carteira_cobranca").coalesce(1).write.mode("overwrite").parquet(
    f"{output_dir_parquet}/obt_carteira_cobranca"
)

# 2. Exportação em CSV
spark.table("obt_funil_originacao").coalesce(1).write.mode("overwrite").option(
    "header", "true"
).csv(f"{output_dir_csv}/obt_funil_originacao")
spark.table("obt_carteira_cobranca").coalesce(1).write.mode("overwrite").option(
    "header", "true"
).csv(f"{output_dir_csv}/obt_carteira_cobranca")

print("Exportações finalizadas com sucesso!")

Exportações finalizadas com sucesso!


## Download dos Arquivos Exportados para a Máquina Local
* Script utilitário em Python para renomear e baixar os arquivos gerados pelo Spark diretamente pelo navegador no Colab.

In [38]:
import glob
import shutil
from google.colab import files

for table in ["obt_funil_originacao", "obt_carteira_cobranca"]:
    parquet_parts = glob.glob(f"{output_dir_parquet}/{table}/*.parquet")
    if parquet_parts:
        target_parquet = f"/content/{table}.parquet"
        shutil.copy(parquet_parts[0], target_parquet)
        print(f"Baixando: {table}.parquet")
        files.download(target_parquet)

    csv_parts = glob.glob(f"{output_dir_csv}/{table}/*.csv")
    if csv_parts:
        target_csv = f"/content/{table}.csv"
        shutil.copy(csv_parts[0], target_csv)
        print(f"Baixando: {table}.csv")
        files.download(target_csv)

Baixando: obt_funil_originacao.parquet


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Baixando: obt_funil_originacao.csv


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Baixando: obt_carteira_cobranca.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>